In [1]:
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


## 1. Load

`data.csv.xls` is not an Excel file despite the name. `file` reports it as
"Non-ISO extended-ASCII text", and the first bytes are `Group;Sex;Age;Pa...`,
so it is semicolon-delimited plain text in Windows-1252. No converter and no
Excel library needed, just the right separator and encoding.

In [ ]:
SOURCE = Path("data.csv.xls")
ENCODING = "cp1252"     # Windows-1252, not UTF-8 (byte 0xde at offset 3270)
SEPARATOR = ";"         # semicolon-delimited, despite the .xls name

TARGET = "mistriage"    # 0 = correct, 1 = under-triage, 2 = over-triage

raw = pd.read_csv(SOURCE, sep=SEPARATOR, encoding=ENCODING)
print(raw.shape)

## 2. Build the frame

The three features are already clean: no nulls, no sentinels, ages 16-96.

`Patients number per hour` is not a feature. It says how many patients each row
stands for, and it is kept for two reasons: you can weight rows by patient volume
later, and `drop_duplicates` will not discard distinct volumes that happen to share
the same age, sex and complaint.

In [ ]:
FEATURES = ["Age", "Sex", "Chief_complain"]
PATIENT_COUNT = "Patients number per hour"   # not a feature: patients per row

df = raw[FEATURES + [TARGET, PATIENT_COUNT]].dropna().copy()
df["Sex"] = df["Sex"].map({1: "Female", 2: "Male"})
df["Chief_complain"] = (
    df["Chief_complain"].str.strip().str.lower().str.replace(r"\s+", " ", regex=True)
)
df = df.drop_duplicates().reset_index(drop=True)   # 13 fully repeated rows
print(f"{len(df):,} cells covering {df[PATIENT_COUNT].sum():,} patients")
df

## 3. Bayesian logistic regression — skeleton

Placedholders are `...` (three dots). Four dots is a `SyntaxError` in Python:
the tokenizer reads the extra `.` as a stray attribute access.

Every reference to `pymc` is inside a comment on purpose, so this cell runs clean
even though `pymc` is not installed. Uncomment the imports and the model block once
you have run `pip install pymc`.

In [ ]:
# =====================================================================
# Bayesian logistic regression: was the patient UNDER-TRIAGED?
#
#     y = 1 if mistriage == 1, else 0
#
# Logistic regression is binary, so a 3-level target has to be turned into one
# yes/no question. We use under-triage, because that is the error that hurts
# patients: 55 cells say yes out of 1,254 (401 patients out of 9,431).
#
# The design mirrors the model in ed_atlas/convert_traces.py:
#     logit(P(outcome)) = beta . x
#     x = sex * cr(age, knots=[43, 65], lower_bound=22, upper_bound=88) * symptoms
#
# NOT INSTALLED: pymc and pytensor are absent from every env on this machine.
# patsy 1.0.1 is already present, which is all the spline basis needs.
#     pip install pymc
# =====================================================================

# import pymc as pm
# import pytensor.tensor as pt
# from patsy import dmatrix        # already installed

SEED = 42
AGE_KNOTS = [22.0, 43.0, 65.0, 88.0]    # same knots as the ED Atlas
AGE_MIN, AGE_MAX = 22.0, 88.0           # the spline is clipped outside this range

# ---------------------------------------------------------------------
# LEAKAGE - do not add any of these as features:
#   mistriage is a deterministic function of (KTAS_RN - KTAS_expert):
#       KTAS_RN == KTAS_expert  ->  mistriage 0
#       KTAS_RN  <  KTAS_expert  ->  mistriage 1  (under-triage)
#       KTAS_RN  >  KTAS_expert  ->  mistriage 2
#   Verified on the frame: 0 exceptions. Error_group == 0 exactly when
#   mistriage == 0. So any of those columns hands over the answer.
#   Length of stay_min and KTAS duration_min are also out: they are recorded
#   after the triage decision, so they do not exist when the decision is made.
# ---------------------------------------------------------------------

X = df[["Age", "Sex", "Chief_complain"]]
y = (df[TARGET] == 1).astype(int)
print(f"binary target: {int(y.sum())} under-triage of {len(y)} cells")

# --- 1. encode ---------------------------------------------------------
# age       -> natural cubic spline basis, 4 columns
# sex       -> 0/1   (published codebook: raw 1 = Female, 2 = Male)
# complaint -> one indicator column per distinct phrasing, 412 of them
#
# The base block is 9 columns: sex, 4 spline, 4 sex:spline. The full
# interaction adds 2 x 412 complaint columns, so the design matrix ends up
# roughly 833 wide against 1,254 rows. That ratio is exactly why this needs
# a prior - a frequentist fit has no hope of estimating 833 coefficients
# from 55 positive cases.
design = ...
# e.g. dmatrix("sex * cr(age, knots=k, lower_bound=lb, upper_bound=ub) * symptoms",
#              {"age": df["Age"], "k": AGE_KNOTS[1:-1],
#               "lb": AGE_MIN, "ub": AGE_MAX, "symptoms": complaint_dummies})

# --- 2. split ----------------------------------------------------------
# stratify=y is not optional: only 55 of 1,254 cells are positive, so an
# unstratified split can leave the test set with almost no positives.
#   X_train, X_test, y_train, y_test = train_test_split(
#       design, y, test_size=0.25, random_state=SEED, stratify=y)
#
# These rows are aggregated cells, not patients: "Patients number per hour"
# runs 1-17 (mean 7.5), so 1,254 rows cover 9,431 patients. There is no
# patient id, so a grouped split is impossible. If you want per-patient
# rather than per-cell inference, weight each row by its patient count:
#   sample_weight = df[PATIENT_COUNT]
train = ...
test = ...

# --- 3. prior ----------------------------------------------------------
# Weakly informative Normal(0, 1.5) on the log-odds scale, one per design
# column, plus a separate intercept. 1.5 keeps the implied odds ratio in a
# plausible range instead of letting 55 positives pull the coefficients
# wherever they like.
beta_prior = ...
alpha_prior = ...

# --- 4. likelihood -----------------------------------------------------
#   with pm.Model() as triage_model:
#       beta  = pm.Normal("beta", mu=0.0, sigma=1.5, shape=n_cols)
#       alpha = pm.Normal("alpha", mu=0.0, sigma=1.5)
#       p     = pm.math.sigmoid(alpha + pt.dot(design_train, beta))
#       y_obs = pm.Bernoulli("y_obs", p=p, observed=y_train)
#       trace = pm.sample(draws=2000, tune=2000, chains=4, target_accept=0.9)
#
# If you want all three levels back, fit three of these, one per value of
# mistriage, which is how ed_atlas fits one model per ICD code.
model = ...
trace = ...

# --- 5. check and predict ----------------------------------------------
#   pm.summary(trace)                 # r_hat, ess_bulk, divergences
#   az.plot_ppc(trace)                # posterior predictive check
#   az.plot_forest(trace)             # credible interval per coefficient
# The posterior is a distribution, not a number: report a credible interval
# for a new patient rather than a hard 0/1, and compare against the base
# rate (55/1254 = 4.4%) so you know the model beats guessing.
prediction = ...

print("skeleton only - nothing has been fitted")